In [3]:
import pandas as pd
import numpy as np

### Load Dataset basic cleaning

In [4]:
df = pd.read_csv("resume_dataset.csv")

# remove missing categories
df = df.dropna(subset=["Category"])

# keep only rows where category is short (likely real domain)
df = df[df["Category"].str.len() < 40]

# convert to uppercase for consistency
df["Category"] = df["Category"].str.upper()

df["Category"].unique()

array(['HR', ' PROCESS IMPROVEMENTS', 'DESIGNER', ' WORD',
       'INFORMATION-TECHNOLOGY', ' ORACLE DATABASE', 'TEACHER',
       'ADVOCATE', ' PROMOTIONS', ' ADDING BENEFICIARIES',
       ' BUSINESS DEVELOPMENT', 'BUSINESS-DEVELOPMENT', 'HEALTHCARE',
       'FITNESS', 'AGRICULTURE', 'BPO', ' AND COBRA', 'SALES',
       'CONSULTANT', 'DIGITAL-MEDIA', ' BUDGETING', ' AD AGENCY',
       'AUTOMOBILE', ' CIM 7.0', 'CHEF',
       ' AND RELATED SUPPLIES SUCH AS ICE', ' WATER', ' AND EVALUATE',
       'FINANCE', '51', ' LUNCH BOX .', 'APPAREL', 'ENGINEERING',
       'ACCOUNTANT', ' SEMI-ANNUAL', ' INVENTORY', 'CONSTRUCTION',
       ' SO-CAL OFFICE TECHNOLOGIES', ' IDAHO', ' BILLINGS',
       'PUBLIC-RELATIONS', ' FOOD AND TABLEWARE', 'BANKING', 'ARTS',
       'AVIATION'], dtype=object)

### Check Domains

In [5]:
df["Category"].value_counts()

Category
INFORMATION-TECHNOLOGY               119
BUSINESS-DEVELOPMENT                 119
AVIATION                             117
CHEF                                 117
FITNESS                              117
ACCOUNTANT                           116
ENGINEERING                          116
FINANCE                              116
SALES                                115
BANKING                              115
ADVOCATE                             115
CONSULTANT                           115
HEALTHCARE                           113
PUBLIC-RELATIONS                     110
HR                                   109
CONSTRUCTION                         108
DESIGNER                             106
ARTS                                 103
TEACHER                              102
APPAREL                               96
DIGITAL-MEDIA                         95
AGRICULTURE                           63
AUTOMOBILE                            35
BPO                                   21
 WATER 

### Choose Only 3 Domains

In [6]:
selected_domains = [
    "INFORMATION-TECHNOLOGY",
    "HR",
    "BUSINESS-DEVELOPMENT"
]

df = df[df["Category"].isin(selected_domains)]

### Reduce Dataset Size

In [7]:
#Use only 10 resumes per domain.

df = df.groupby("Category").head(10)

### Create Resume Dictionaries

In [8]:
#Convert dataset into dictionaries

resumes = {}
resume_categories = {}

for i, row in df.iterrows():

    resume_id = f"resume_{i}"

    resumes[resume_id] = row["Resume_str"]
    resume_categories[resume_id] = row["Category"]

### Define Job Descriptions

In [9]:
job_descriptions = {

"INFORMATION-TECHNOLOGY": """
Software Engineer

Required Skills:
Python
Programming
Algorithms
Software development
Databases
""",

"HR": """
HR Manager

Required Skills:
Recruitment
Employee relations
HR operations
Performance management
""",

"BUSINESS-DEVELOPMENT": """
Business Development Manager

Required Skills:
Sales strategy
Client acquisition
Market analysis
Lead generation
Negotiation
"""
}

### Filtering Step

In [10]:
selected_jd = "INFORMATION-TECHNOLOGY"

jd_text = job_descriptions[selected_jd]

filtered_resumes = {}

for name, text in resumes.items():

    if resume_categories[name] == selected_jd:
        filtered_resumes[name] = text

### Install Required Libraries

In [ ]:
#!pip install sentence-transformers scikit-learn langchain-text-splitters

### Load Embedding Model

In [12]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

model = SentenceTransformer("all-MiniLM-L6-v2")



#This model converts text into vector embeddings

c:\Users\Asus\anaconda3\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


### Convert JD to Embedding

In [13]:
jd_embedding = model.encode([jd_text])

#Now the job description is represented as a vector

### Rank Filtered Resumes

In [14]:
scores = []

for name, resume_text in filtered_resumes.items():

    resume_embedding = model.encode([resume_text])

    similarity = cosine_similarity(jd_embedding, resume_embedding)[0][0]

    scores.append((name, similarity))


#Now compute similarity between the JD and each resume

### Sort Candidates

In [15]:
scores = sorted(scores, key=lambda x: x[1], reverse=True)

### Show Top Candidates

In [16]:
for rank, (name, score) in enumerate(scores[:5], start=1):

    print(f"Rank {rank}: {name} — Score {round(score*100,2)}%")

Rank 1: resume_224 — Score 34.17%
Rank 2: resume_219 — Score 32.59%
Rank 3: resume_222 — Score 32.19%
Rank 4: resume_227 — Score 31.32%
Rank 5: resume_223 — Score 31.14%


### RAG Step (Chunk the Resume)

In [17]:
#Instead of comparing whole resumes, split them into chunks

from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=100
)

In [18]:
top_resume_name = scores[0][0]

top_resume_text = filtered_resumes[top_resume_name]

chunks = splitter.split_text(top_resume_text)

print(len(chunks))

19


### Retrieve Most Relevant Resume Sections

In [19]:
chunk_embeddings = model.encode(chunks)

similarities = cosine_similarity(jd_embedding, chunk_embeddings)[0]

In [20]:
#Find best chunks:

import numpy as np

top_indices = np.argsort(similarities)[-3:]

top_chunks = [chunks[i] for i in top_indices]


#These are the most relevant resume sections.
#This is the Retrieval step in RAG.

### Install LLM for GenAI

In [ ]:
#!pip install langchain-groq

### Load Llama Model

In [21]:
from langchain_groq import ChatGroq

llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0
)

### Generate AI Evaluation

In [22]:
context = "\n\n".join(top_chunks)

prompt = f"""
You are an HR recruiter.

Job Description:
{jd_text}

Candidate Resume Sections:
{context}

Evaluate the candidate and provide:

1. Match score (0–100)
2. Matching skills
3. Missing skills
4. Recommendation
"""

### Get AI Response

In [23]:
response = llm.invoke(prompt)

print(response.content)

Based on the job description and the candidate's resume, here's the evaluation:

**Match Score: 60**

The candidate has some relevant skills, but they don't directly match the required skills for the Software Engineer position. However, they do have some transferable skills that could be valuable in a software development role.

**Matching Skills:**

1. **Programming**: The candidate has experience with hardware and software installation, patching, and configuration, which implies some programming skills, although not explicitly mentioned.
2. **Complex problem-solving**: The candidate has experience with problem-solving, which is a valuable skill for a software engineer.
3. **Systematic focus**: The candidate has a systematic approach to their work, which is essential for software development.

**Missing Skills:**

1. **Python**: The candidate does not mention Python programming skills, which is a required skill for the Software Engineer position.
2. **Algorithms**: There is no mention